# Reproducing Kim et al. (2023) from dandiset 000633

**Paper.** Kim V, Semenov I, Kiester AS, Keppler MA, Ibey BL, Bixler JN, Colunga Biancatelli RML,
Pakhomov AG (2023). Control of the electroporation efficiency of nanosecond pulses by swinging the
electric field vector direction. *Int. J. Mol. Sci.* 24, 10921.
[doi:10.3390/ijms241310921](https://doi.org/10.3390/ijms241310921).

**Claims file.** `claims/000633/10.3390-ijms241310921.yaml`, written from the version of record
(open access, Europe PMC PMC10341945). The paper has no supplement and links no code.

**Dandiset.** [DANDI:000633](https://dandiarchive.org/dandiset/000633/0.231013.2226), version
0.231013.2226. The paper cites no version of the dandiset. This is the only published version, and
the most recent; the draft lists the same two assets with the same sizes and was last modified
when this version was published (2023-10-13), so the two do not differ in anything the analysis uses.

**What the dandiset holds.** Two NWB files, each one stitched whole-dish fluorescence image of a
BPAE cell monolayer in two channels (MemBrite and YO-PRO-1): `Single-pulse_trains` and
`Paired-pulse_trains`. The dandiset's title is the legend of Figure 4 ("The difference in
electroporation patterns produced by a train of single pulses and a train of paired pulses"), so
these are the images of Figure 4A (single pulses) and Figure 4B (paired pulses). The ROI
measurements of Figure 2 (n = 3 at each of 1 Hz and 833 kHz), the square-array experiments of
Figure 3 and the strobe imaging of Figure 5 are not in it.

**Outside inputs.** None. Two claims need the electric field map of the paper's Sim4Life
simulation (Methods 4.3), which is not deposited. It is computed here from the geometry the
paper states and the electrode positions measured in the images (see `swing.py`); this
substitutes for the paper's solver and is recorded as a deviation.

| Claim | Tested here |
|---|---|
| fig2-cancellation-at-180 | Blocked: data missing (Figure 2 data not deposited) |
| fig2-crossover-angle | Blocked: data missing |
| fig2-facilitation-plateau | Blocked: data missing |
| fig2-60-fold-co-counter | Blocked: data missing |
| fig3-1hz-co-cross-same | Blocked: data missing (Figure 3 data not deposited) |
| fig3-770khz-co-enhanced | Blocked: data missing |
| fig3-770khz-cross-same-as-1hz | Blocked: data missing |
| fig4a-single-pulse-follows-field | Tested |
| fig4bc-paired-pulse-focal-apex | Tested |
| fig4-comet-shape | Tested |
| fig5-base-sawtooth | Blocked: data missing (Figure 5 data not deposited) |
| fig5-apex-larger-polarization | Blocked: data missing |

For the Figure 2 claims, a section near the end computes, for information only, the 833 kHz
angle curve that the two Figure 4 images allow (one dish each, not the paper's n = 3, and no
1 Hz data). It decides no outcome.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from scipy.ndimage import median_filter, uniform_filter

import swing

pd.set_option("display.width", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
assets = swing.assets()
meta, img = {}, {}
for a in assets:
    m, im = swing.load_binned(a["url"])
    m["path"], m["size"] = a["path"], a["size"]
    meta[m["session_id"]], img[m["session_id"]] = m, im
SINGLE, PAIRED = "Single-pulse_trains", "Paired-pulse_trains"
print(f"dandiset {swing.DANDISET_ID} version {swing.DANDISET_VERSION}: {len(assets)} files streamed "
      f"in {time.time() - started:.0f} s, binned {swing.BIN} x {swing.BIN}")

## Data check

Each check of the description against the values is stated under its output.

In [ ]:
for s in (SINGLE, PAIRED):
    m = meta[s]
    print(f"== {m['path']} ({m['size'] / 1e9:.2f} GB, {m['bytes_fetched'] / 1e9:.2f} GB fetched)")
    print("  session_id / subject_id:", m["session_id"], "/", m["subject_id"], "| strain:", m["strain"])
    print("  data shape", m["shape"], m["dtype"], "chunks", m["chunks"], "unit", repr(m["unit"]),
          "ImageSeries rate", m["rate"], "Hz")
    for c, st in enumerate(m["full_stats"]):
        print(f"  channel {c}: min {st['min']}, max {st['max']}, mean {st['mean']:.1f}, "
              f"fraction of zero pixels {st['zero_fraction']:.3f}")
    print("  session_description:", m["session_description"])
print("\nimage description (same in both files):", meta[SINGLE]["image_description"])

The image is stored as an `ImageSeries` of shape (22323, 22323, 2) with a rate of 2 Hz. NWB
reads the first axis of an `ImageSeries` as time, so a standard reader sees 22323 frames of
22323 x 2 pixels; the data are one image of 22323 x 22323 pixels in two channels, as the
description of stitched tiles says. No pixel size, channel names or intensity unit (`n.a.`) are
stored; the channel order comes only from the free-text description (MemBrite listed first, then
YO-PRO-1), and is checked below. About 16% of pixels are exactly zero in every channel: the
corners outside the scanned tiles.

The session description of `Single-pulse_trains` describes another experiment: "4 co-directional
paired-pulse (600 + 600 ns) trains ... 770 kHz, amplitude 10 kV/cm", which are the settings of
Figure 3 (Section 2.2; there, a square array of four electrodes). The image shows three electrode
footprints in a triangle (below), as in Figure 4A. The file is taken as Figure 4A from its name
and the dandiset's title. The `Paired-pulse_trains` description gives an amplitude of 2 kV, while
the paper gives 4 kV for both Figure 4 images (Section 2.3); the field check below bears on this.
The description mentions a six-well plate; the paper used 35 mm glass-bottom dishes (Methods 4.1).
Subject age is "P1D" for a cell line, and each "subject" is a condition, not an animal.

In [ ]:
# Electrode footprints and image scale (swing.find_footprints), and channel identity.
fps = {s: swing.find_footprints(img[s][1]) for s in (SINGLE, PAIRED)}
rows = []
for s in (SINGLE, PAIRED):
    for f in fps[s]:
        rows.append({"image": s, "x_bin": round(f["x"], 1), "y_bin": round(f["y"], 1), "radius_bin": round(f["r"], 1),
                     "edge_sd_bin": round(f["edge_sd"], 1), "edge_points_of_72": f["edge_points"],
                     "coarse_contrast": round(f["contrast"], 3)})
fp_table = pd.DataFrame(rows)
print(fp_table.to_string(index=False))

r_paired = np.mean([f["r"] for f in fps[PAIRED]])
um = swing.NEEDLE_OD_UM / (2 * r_paired)
print(f"\nScale from the paired-image footprints (their edges are sharp; edge sd {fp_table[fp_table.image == PAIRED].edge_sd_bin.max()} bins or less):")
print(f"  mean footprint diameter {2 * r_paired:.1f} binned px = 1.5 mm  ->  {um:.2f} um per binned px, "
      f"{um / swing.BIN:.4f} um per full-resolution px")
print("  ORCA-Flash4 (6.5 um pixels) behind a 10x objective gives 0.65 um per pixel")
print(f"  image width: {meta[SINGLE]['shape'][1] * um / swing.BIN / 1000:.2f} mm")
r_single = np.mean([f["r"] for f in fps[SINGLE]])
print(f"  single-image footprints fitted to a mean diameter of {2 * r_single * um / 1000:.2f} mm (edges less sharp)")

E = {}
for s in (SINGLE, PAIRED):
    apex, ba, bb = swing.label_electrodes(fps[s])
    E[s] = {"apex": apex, "base_a": ba, "base_b": bb}
    P = lambda f: np.array([f["x"], f["y"]]) * um
    print(f"\n{s}: apex at ({apex['x']:.0f}, {apex['y']:.0f}), base electrodes at ({ba['x']:.0f}, {ba['y']:.0f}) "
          f"and ({bb['x']:.0f}, {bb['y']:.0f}) binned px")
    print(f"  base side {np.linalg.norm(P(ba) - P(bb)) / 1000:.2f} mm, sides to apex "
          f"{np.linalg.norm(P(apex) - P(ba)) / 1000:.2f} and {np.linalg.norm(P(apex) - P(bb)) / 1000:.2f} mm")

print("\nFootprint contrast by channel (median inside the footprint cores / median of a ring 0.2 to 0.6 mm outside the edges):")
for s in (SINGLE, PAIRED):
    for c in range(2):
        ch = img[s][c]
        inner = np.concatenate([ch[swing.disc_mask(ch.shape, f["x"], f["y"], swing.FOOTPRINT_CORE * 750 / um)] for f in fps[s]])
        ring = np.concatenate([ch[swing.disc_mask(ch.shape, f["x"], f["y"], (750 + 600) / um)
                                  & ~swing.disc_mask(ch.shape, f["x"], f["y"], (750 + 200) / um)] for f in fps[s]])
        print(f"  {s} channel {c}: {np.median(inner):.0f} / {np.median(ring):.0f} = {np.median(inner) / np.median(ring):.2f}")

a, b = img[SINGLE][1].ravel(), img[PAIRED][1].ravel()
print("\nCorrelation of the two YP images, binned pixel by pixel:", round(float(np.corrcoef(a, b)[0, 1]), 3))

Both images show three dark footprints in an isosceles triangle with a base of 4.6 to 4.7 mm and
sides of 6.6 to 6.8 mm, so the two dishes used the same array, turned about 180 degrees between
them. The scale set from the 1.5 mm needle (Methods 4.2) is 0.654 um per pixel, which is what the
camera and objective named in the file give (6.5 um / 10 = 0.65 um), and makes the image 14.6 mm
wide, about the size of a 14 mm glass window. The scale is consistent with the data, not shown by
them. Footprints are dark in channel 1, where cells under the needle were shielded from the field
and took up no dye (ratio 0.68 and 0.81), and not in channel 0 (0.99 and 1.03); channel 1 is YO-PRO-1 and channel 0 MemBrite,
as the description lists them. The two images are distinct recordings (different footprint
positions, low pixel correlation), so neither is deposited twice. They are still images: there
are no time series or trial tables whose clocks need to agree.

In [ ]:
# The field model (swing.solve_potential): 2D conduction between three 1.5 mm electrodes in a 35 mm dish.
field = {}
for s in (SINGLE, PAIRED):
    e = E[s]
    els = [(f["x"] * um, f["y"] * um) for f in (e["apex"], e["base_a"], e["base_b"])]
    ny, nx = img[s][1].shape
    t0 = time.time()
    origin, h, phi = swing.solve_potential(els, [[0, 1, 0], [0, 0, 1]], (nx / 2 * um, ny / 2 * um))
    field[s] = {"origin": origin, "h": h, "phi": phi}
    for k in range(2):
        ex, ey = swing.field_from_potential(phi[k], h)
        field[s][f"e{k}"] = (ex, ey)
        field[s][f"mag{k}"] = np.hypot(ex, ey) * 1e4          # V/cm per volt applied
    print(f"{s}: solved on a {phi.shape[1]} x {phi.shape[2]} grid of {h:.0f} um in {time.time() - t0:.0f} s")

# The paper's field along the bisecting line (Section 2.1: 5 to 10 kV/cm) against the amplitude.
e = E[PAIRED]
bx, by, bd = swing.bisector_points(e["apex"], (e["base_a"], e["base_b"]), um)
mag_b = swing.sample(field[PAIRED]["mag0"], field[PAIRED]["origin"], field[PAIRED]["h"], bx * um, by * um)
print(f"\nField on the bisecting line of the paired-pulse dish, {len(bd)} points from the base midpoint to the apex footprint:")
for volts in (4000, 2000):
    print(f"  at {volts / 1000:.0f} kV: {mag_b.min() * volts / 1000:.1f} to {mag_b.max() * volts / 1000:.1f} kV/cm")

# Grid check: the same solve on a 60 um grid.
els = [(f["x"] * um, f["y"] * um) for f in (e["apex"], e["base_a"], e["base_b"])]
ny, nx = img[PAIRED][1].shape
o60, h60, phi60 = swing.solve_potential(els, [[0, 1, 0]], (nx / 2 * um, ny / 2 * um), h=60.0)
ex60, ey60 = swing.field_from_potential(phi60[0], h60)
m60 = swing.sample(np.hypot(ex60, ey60) * 1e4, o60, h60, bx * um, by * um)
print(f"  60 um grid against 40 um grid on the bisecting line: largest relative difference {np.max(np.abs(m60 / mag_b - 1)):.3f}")

With the 4 kV of Section 2.3, the computed field on the bisecting line runs from about 4 to
10 kV/cm, close to the "5 to 10 kV/cm" Section 2.1 gives for this array; with the 2 kV of the
file's description it would be 2 to 5 kV/cm. This is consistent with the paper's amplitude, not a
proof of it: the field model is computed here, not measured. The solve changes little with the grid.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
for ax, s in zip(axes, (SINGLE, PAIRED)):
    yp = median_filter(img[s][1], 3)
    lo, hi = np.percentile(yp[yp > 0], [1, 99.5])
    ext = [0, yp.shape[1] * um / 1000, yp.shape[0] * um / 1000, 0]
    ax.imshow(yp, cmap="gray", vmin=lo, vmax=hi, extent=ext)
    for name, f in E[s].items():
        ax.add_patch(plt.Circle((f["x"] * um / 1000, f["y"] * um / 1000), 0.75, fill=False, color="tab:orange", lw=1))
        ax.annotate(name, (f["x"] * um / 1000, f["y"] * um / 1000), color="tab:orange", ha="center", va="center", fontsize=8)
    ax.set_title(f"{s}, YO-PRO-1 (display {lo:.0f} to {hi:.0f})")
    ax.set_xlabel("mm"); ax.set_ylabel("mm")
plt.tight_layout(); plt.show()
print("Display range clips the brightest 0.5% of binned pixels in each panel.")

## fig4a-single-pulse-follows-field

> Under a train of five single pulses applied to one base electrode (833 kHz, 4 kV), YP uptake is
> distributed as the electric field strength map.

Criterion (claims file): background-subtracted YP in 200 x 200 um ROIs over the image (footprint
background, Methods 4.4), compared with the field in the same ROIs by (a) Spearman rank
correlation or (b) the order of the three electrode neighbourhoods by YP matching their order by
field. Reproduced when both agree with the map (correlation positive with p < 0.05, and the same
order).

The field map is computed here (see the data check), for a single pulse on one base electrode
with the other two grounded (Methods 4.2), scaled to 4 kV. Which base electrode is E1, the one
pulsed, is not in the file; both are run. The neighbourhood of an electrode is taken as the ROIs
whose centre lies within 500 um of its footprint edge (the claims file does not fix it; 1000 um
is shown too). The ROI grid covers the image within 6 mm of its centre, leaving out ROIs whose
centre is within 150 um of a footprint edge; the effect of both choices is shown.

In [ ]:
s = SINGLE
e = E[s]
yp = img[s][1].astype(float)
bg, bg_each = swing.background(yp, fps[s], 750 / um)
ypb = yp - bg
print(f"Footprint background: {bg:.1f} (per footprint {np.round(bg_each, 1).tolist()}); dish median {np.median(yp[yp > 0]):.1f}")
names = ["apex", "base_a", "base_b"]
assign = {"E1 = base_a (right in the image)": 0, "E1 = base_b (left in the image)": 1}

def fig4a_measure(radius_um, margin_um, ring_um, k):
    cen = swing.roi_grid(yp.shape, fps[s], um, radius_um, margin_um)
    Y = np.array([swing.roi_mean(ypb, x, y, swing.ROI_UM / um) for x, y in cen])
    F = swing.roi_field(field[s][f"mag{k}"] * 4000 / 1000, field[s]["origin"], field[s]["h"], cen, um)  # kV/cm
    rho, p = stats.spearmanr(F, Y)
    neigh = []
    for n in names:
        f = e[n]
        d = np.hypot(cen[:, 0] - f["x"], cen[:, 1] - f["y"]) * um
        m = d < 750 + ring_um
        neigh.append((n, Y[m].mean(), F[m].mean(), int(m.sum())))
    order_yp = [x[0] for x in sorted(neigh, key=lambda t: -t[1])]
    order_f = [x[0] for x in sorted(neigh, key=lambda t: -t[2])]
    return len(cen), rho, p, neigh, order_yp, order_f

rows = []
for label, k in assign.items():
    n, rho, p, neigh, oy, of = fig4a_measure(6000, 150, 500, k)
    print(f"\n{label}: {n} ROIs, Spearman rho = {rho:.3f}, p = {p:.2g}")
    for name, y, fld, cnt in neigh:
        print(f"   {name:7s} neighbourhood (0-500 um): {cnt} ROIs, mean YP above background {y:7.1f}, mean field {fld:5.2f} kV/cm")
    print(f"   order by YP: {oy}; order by field: {of}; same order: {oy == of}")
    for radius in (5000, 6000):
        for margin in (150, 400):
            for ring in (500, 1000):
                n2, r2, p2, _, oy2, of2 = fig4a_measure(radius, margin, ring, k)
                rows.append({"assignment": label, "grid_radius_mm": radius / 1000, "margin_um": margin, "neighbourhood_um": ring,
                             "rois": n2, "spearman_rho": round(r2, 3), "p": float(f"{p2:.2g}"), "same_order": oy2 == of2})
print("\nHow the choices change the result:")
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
cen = swing.roi_grid(yp.shape, fps[s], um)
Y = np.array([swing.roi_mean(ypb, x, y, swing.ROI_UM / um) for x, y in cen])
for ax, (label, k) in zip(axes[:2], assign.items()):
    F = swing.roi_field(field[s][f"mag{k}"] * 4, field[s]["origin"], field[s]["h"], cen, um)
    ax.scatter(F, Y, s=4, alpha=0.4, color="tab:blue")
    ax.set_xscale("log"); ax.set_xlabel("computed field in the ROI (kV/cm, 4 kV)"); ax.set_ylabel("YP above footprint background")
    ax.set_title(label, fontsize=10)
axes[2].imshow(np.log10(field[s]["mag0"] * 4), cmap="viridis", vmin=0, vmax=1.3)
axes[2].set_title("log10 field (kV/cm), E1 = base_a, field grid", fontsize=10); axes[2].axis("off")
plt.tight_layout(); plt.show()
print(f"YP per ROI ranges from {Y.min():.0f} to {Y.max():.0f}; field per ROI {F.min():.2f} to {F.max():.2f} kV/cm; all points are inside the axes.")

**Result: reproduced, resting on the assumption that E1 is the right-hand base electrode
(`base_a`).** With E1 = `base_a`, YP and the computed field are rank-correlated over the ROIs
(rho and p printed above), and the three neighbourhoods have the same order by YP and by field
(base_a, then base_b, then the apex). With E1 = `base_b` the correlation is still positive and
significant but weaker, and the order differs (YP is highest at `base_a`, the field at `base_b`),
so the criterion's two readings would disagree. The file does not say which base electrode was
pulsed. The assignment is taken from the pattern itself, which makes the test partly circular:
what the data show without the assumption is a positive rank correlation under either
assignment, and an order of neighbourhoods that matches the field map for one of the two. The
table of choices shows how the correlation and order hold across grid radius, margin and
neighbourhood width.

## fig4bc-paired-pulse-focal-apex

> Under five paired pulses applied to the two base electrodes in alternation (833 kHz, 4 kV),
> electroporation is strong near the grounded apex electrode and weak or absent near the base
> electrodes, despite the strong field there.

Criterion (claims file): background-subtracted YP in 200 x 200 um ROIs centred (a) 100 um and
(b) 500 um outside the edge of each footprint, on the line from that electrode toward the array
centre. Reproduced when, under both readings, YP near the apex in the paired-pulse image exceeds
that near each base electrode, and the ratio of apex to base YP is higher in the paired-pulse
image than in the single-pulse image. The array centre is taken as the centroid of the three
electrode centres. The footprint edge is 750 um from the fitted centre (Methods 4.2).

In [ ]:
near = {}
for s in (SINGLE, PAIRED):
    e = E[s]
    yp = img[s][1].astype(float)
    bg, _ = swing.background(yp, fps[s], 750 / um)
    cen = np.mean([[e[n]["x"], e[n]["y"]] for n in names], axis=0)
    for d in (100, 500):
        for n in names:
            f = e[n]
            v = cen - np.array([f["x"], f["y"]]); v /= np.linalg.norm(v)
            x, y = np.array([f["x"], f["y"]]) + v * (750 + d) / um
            near[(s, d, n)] = (swing.roi_mean(yp - bg, x, y, swing.ROI_UM / um), swing.roi_mean(yp, x, y, swing.ROI_UM / um))
rows = []
for d in (100, 500):
    for s in (SINGLE, PAIRED):
        a = near[(s, d, "apex")][0]
        rows.append({"distance_um": d, "image": s, **{f"YP_{n}": round(near[(s, d, n)][0], 1) for n in names},
                     "apex/base_a": round(a / near[(s, d, "base_a")][0], 3), "apex/base_b": round(a / near[(s, d, "base_b")][0], 3)})
tab = pd.DataFrame(rows)
print("YP above the footprint background in the ROIs near each electrode:")
print(tab.to_string(index=False))
for d in (100, 500):
    P = tab[(tab.distance_um == d) & (tab.image == PAIRED)].iloc[0]
    S = tab[(tab.distance_um == d) & (tab.image == SINGLE)].iloc[0]
    print(f"\n{d} um: paired apex > each base: {P.YP_apex > P.YP_base_a and P.YP_apex > P.YP_base_b}; "
          f"apex/base higher in paired than single: base_a {P['apex/base_a'] > S['apex/base_a']}, base_b {P['apex/base_b'] > S['apex/base_b']}")
print("\nVariant without background subtraction (raw YP), paired image:")
for d in (100, 500):
    print(f"  {d} um: " + ", ".join(f"{n} {near[(PAIRED, d, n)][1]:.1f}" for n in names))
print("Field at these ROIs in the paired dish (kV/cm at 4 kV, pulse on base_a / on base_b):")
for d in (100, 500):
    e = E[PAIRED]; cen = np.mean([[e[n]["x"], e[n]["y"]] for n in names], axis=0)
    out = []
    for n in names:
        f = e[n]; v = cen - np.array([f["x"], f["y"]]); v /= np.linalg.norm(v)
        x, y = (np.array([f["x"], f["y"]]) + v * (750 + d) / um) * um
        m0 = swing.sample(field[PAIRED]["mag0"] * 4, field[PAIRED]["origin"], field[PAIRED]["h"], x, y)[0]
        m1 = swing.sample(field[PAIRED]["mag1"] * 4, field[PAIRED]["origin"], field[PAIRED]["h"], x, y)[0]
        out.append(f"{n} {m0:.1f}/{m1:.1f}")
    print(f"  {d} um: " + ", ".join(out))

**Result: reproduced.** Under both readings, YP near the apex in the paired-pulse image is above
YP near each base electrode (2.7 to 8.1 times, from the table), and the apex-to-base ratios are
far higher in the paired-pulse image than in the single-pulse image, where the apex
neighbourhood has the least uptake (ratios 0.05 to 0.20). The computed field near the base
electrodes during the pulse on that electrode is higher than near the apex, as the claim says
("despite the strong field there").

## fig4-comet-shape

> Under paired pulses the electroporated cells form a comet-shaped region pointing from the apex
> electrode towards the base.

Criterion (claims file): the electroporated region is background-subtracted YP above (a) half
and (b) a quarter of its maximum in the paired-pulse image, after smoothing over 200 x 200 um.
Reproduced when, under both readings, the region touches the apex footprint and extends further
along the bisecting line toward the base than across it.

The region is the set of connected pixels above threshold that touch the apex footprint
(`swing.comet_region`), inside 6 mm of the image centre and outside the footprints. "Further
toward the base than across" is measured from the apex centre: the reach along the bisecting line
toward the base against the largest distance across it, on either side. The extents along and
across the line are shown too. How the maximum is taken matters (below), so three versions are
run: the literal maximum of the 200 um mean, the 99.5th percentile of the 200 um mean, and the
maximum of a 200 um median filter.

In [ ]:
s = PAIRED
e = E[s]
yp = img[s][1].astype(float)
bg, _ = swing.background(yp, fps[s], 750 / um)
ypb = yp - bg
ny, nx = yp.shape
Yg, Xg = np.mgrid[:ny, :nx]
valid = np.hypot(Xg - nx / 2, Yg - ny / 2) * um < 6000
for n in names:
    valid &= ~swing.disc_mask(yp.shape, e[n]["x"], e[n]["y"], 750 / um)
k = int(round(swing.ROI_UM / um))
sm_mean = uniform_filter(ypb, k)
sm_med = median_filter(ypb, k)
iy, ix = np.unravel_index(np.argmax(np.where(valid, sm_mean, -np.inf)), sm_mean.shape)
print(f"Smoothing window {k} binned px ({k * um:.0f} um).")
print(f"Literal maximum of the 200 um mean: {sm_mean[iy, ix]:.0f} at ({ix}, {iy}) binned px; "
      f"next highest away from it (outside 500 um): "
      f"{np.max(np.where(valid & (np.hypot(Xg - ix, Yg - iy) * um > 500), sm_mean, -np.inf)):.0f}")
full_px = img[s][1][iy - 3:iy + 4, ix - 3:ix + 4]
print(f"  binned YP around it: max {full_px.max():.0f}, against a 99.9th percentile of binned YP in the dish of {np.percentile(yp[valid], 99.9):.0f}")
print(f"  distance from the apex centre {np.hypot(ix - e['apex']['x'], iy - e['apex']['y']) * um:.0f} um")
versions = {"mean, literal max": (sm_mean, sm_mean[valid].max()),
            "mean, 99.5th percentile": (sm_mean, np.percentile(sm_mean[valid], 99.5)),
            "median filter, max": (sm_med, sm_med[valid].max())}
rows, regions = [], {}
for vname, (im, M) in versions.items():
    for frac in (0.5, 0.25):
        R, o = swing.comet_region(im, frac * M, e["apex"], (e["base_a"], e["base_b"]), um, valid)
        regions[(vname, frac)] = R
        ok = o["touches_apex"] and o["pixels"] > 0 and o["reach_toward_base"] > o["reach_across"]
        rows.append({"version": vname, "max": round(M), "fraction": frac, "area_mm2": round(o["area_mm2"], 2),
                     "touches_apex": o["touches_apex"],
                     **{c: float(np.round(o.get(c, np.nan))) for c in ("reach_toward_base", "reach_away", "reach_across", "u_extent", "v_extent")},
                     "touches_base": o.get("touches_base"), "toward_base>across": ok})
print("\nDistances in um, from the apex centre (u along the bisecting line toward the base, v across it):")
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
ext = [0, nx * um / 1000, ny * um / 1000, 0]
for ax, vname in zip(axes, ("mean, 99.5th percentile", "median filter, max")):
    im, M = versions[vname]
    ax.imshow(np.where(valid, im, np.nan), cmap="gray", extent=ext, vmin=0, vmax=M)
    for frac, col in ((0.5, "tab:orange"), (0.25, "tab:blue")):
        ax.contour(np.linspace(0, nx * um / 1000, nx), np.linspace(0, ny * um / 1000, ny), regions[(vname, frac)].astype(float),
                   levels=[0.5], colors=col, linewidths=1)
    ax.plot([e["apex"]["x"] * um / 1000, (e["base_a"]["x"] + e["base_b"]["x"]) / 2 * um / 1000],
            [e["apex"]["y"] * um / 1000, (e["base_a"]["y"] + e["base_b"]["y"]) / 2 * um / 1000], ls="--", color="white", lw=0.8)
    ax.set_title(f"{vname}: half (orange) and quarter (blue) maximum", fontsize=10)
    ax.set_xlabel("mm"); ax.set_ylabel("mm")
plt.tight_layout(); plt.show()
print("Grey scale 0 to the maximum of each version; values above it are shown at the top of the scale.")

**Result: reproduced, resting on the assumption that the maximum is taken robustly to isolated
bright objects.** With the literal maximum of the 200 um mean, the maximum lies on one very
bright object (printed above, far brighter than the rest of the dish), and the half-maximum
region is empty; the quarter-maximum region is a few pixels at the apex footprint, behind it,
so the literal reading does not give a region that can be judged. With the 99.5th percentile of
the 200 um mean, or the maximum of a 200 um median filter, the region at both half and quarter
maximum touches the apex footprint and reaches further toward the base than across the line
(table above), and reaches less far away from the base than toward it; it touches neither base
footprint.

## Figure 2 at 833 kHz from the Figure 4 images (for information; decides no outcome)

The Figure 2 claims need the ROI data of Figure 2: three experiments at each of 1 Hz and 833 kHz.
They are not in the dandiset, and the four Figure 2 claims are blocked. The two Figure 4 images
are one dish each of five single 600 ns pulses and of five paired pulses at 833 kHz in the same
array, so they allow one 833 kHz curve with n = 1, if they were made in the way of Section 2.1.
The paper does not say they were. The curve is computed as Section 2.1 describes, for information:
contiguous 200 um ROIs on the bisecting line from the base midpoint to the apex footprint, YP above
the footprint background, the paired-pulse value as % of the single-pulse value at the same
distance from the base midpoint, and the angle between the field vectors of the two pulses of a
pair from the computed field. The paper's 32 ROIs do not fit on this line (5.5 mm, 27 ROIs).

In [ ]:
prof = {}
for s in (SINGLE, PAIRED):
    e = E[s]
    yp = img[s][1].astype(float)
    bg, _ = swing.background(yp, fps[s], 750 / um)
    x, y, d = swing.bisector_points(e["apex"], (e["base_a"], e["base_b"]), um)
    prof[s] = (d, np.array([swing.roi_mean(yp - bg, xx, yy, swing.ROI_UM / um) for xx, yy in zip(x, y)]), x, y)
dP, P, xP, yP = prof[PAIRED]; dS, S, _, _ = prof[SINGLE]
n = min(len(P), len(S))
F = field[PAIRED]
f1 = tuple(swing.sample(c, F["origin"], F["h"], xP * um, yP * um) for c in F["e0"])
f2 = tuple(swing.sample(c, F["origin"], F["h"], xP * um, yP * um) for c in F["e1"])
angle = swing.vector_angle(f1, f2)
curve = pd.DataFrame({"distance_um": dP[:n], "angle_deg": angle[:n].round(1), "single_YP": S[:n].round(1),
                      "paired_YP": P[:n].round(1), "paired_pct_of_single": (100 * P[:n] / S[:n]).round(1)})
print(curve.to_string(index=False))
r180 = curve.paired_pct_of_single.iloc[0]
fit = np.polyfit(curve.angle_deg[curve.angle_deg > 140], curve.paired_pct_of_single[curve.angle_deg > 140], 2)
print(f"\n180 degrees: ROI nearest the base ({curve.angle_deg.iloc[0]} deg) {r180:.1f}% -> {100 / r180:.1f}-fold weaker than single pulses "
      f"(paper 20-fold at 833 kHz); quadratic fit over angles above 140 deg, at 180: {np.polyval(fit, 180):.1f}%")
pc = curve.paired_pct_of_single.values; ang = curve.angle_deg.values
cross = [i for i in range(len(pc) - 1) if (pc[i] - 100) * (pc[i + 1] - 100) < 0]
print("Crossings of 100% (angle by linear interpolation):",
      [round(float(ang[i] + (100 - pc[i]) * (ang[i + 1] - ang[i]) / (pc[i + 1] - pc[i])), 1) for i in cross], "(paper about 130 deg at 833 kHz)")
band = curve[(curve.angle_deg >= 90) & (curve.angle_deg <= 100)]
print(f"90 to 100 deg: {len(band)} ROI(s), mean {band.paired_pct_of_single.mean():.0f}% (paper about 300%)")
plateau = curve[curve.angle_deg < 90]
print(f"Below 90 deg: {len(plateau)} ROIs, mean {plateau.paired_pct_of_single.mean():.0f}%, median {plateau.paired_pct_of_single.median():.0f}%; "
      f"single-pulse YP there {plateau.single_YP.min():.1f} to {plateau.single_YP.max():.1f} above background, so the ratio is unstable")
print(f"Co- over counter-directional (paper 60-fold): plateau mean / 180 = {plateau.paired_pct_of_single.mean() / r180:.0f}, "
      f"ROI nearest 0 deg ({ang[-1]} deg) / 180 = {pc[-1] / r180:.0f}")

These numbers come from one pair of dishes that the paper presents as Figure 4, not from the
Figure 2 experiments, and they decide no outcome. Near the base the single-pulse YP is high and
the ratio is stable; toward the apex the single-pulse YP falls close to the footprint
background, so the ratio there swings widely from one ROI to the next.

## Blocked claims

- **fig2-cancellation-at-180, fig2-crossover-angle, fig2-facilitation-plateau,
  fig2-60-fold-co-counter**: data missing. The 32-ROI measurements of Figure 2, three experiments
  at each of 1 Hz and 833 kHz, are not in the dandiset; there are no 1 Hz images at all. The
  Sim4Life field simulation that gives each ROI's angle is not deposited either.
- **fig3-1hz-co-cross-same, fig3-770khz-co-enhanced, fig3-770khz-cross-same-as-1hz**: data
  missing. No image from the square four-electrode array of Figure 3 is in the dandiset (both
  images show three footprints).
- **fig5-base-sawtooth, fig5-apex-larger-polarization**: data missing. The strobe images of
  FluoVolt-loaded CHO cells are not in the dandiset.

## Summary

| Claim | Outcome |
|---|---|
| fig2-cancellation-at-180 | Blocked: data missing |
| fig2-crossover-angle | Blocked: data missing |
| fig2-facilitation-plateau | Blocked: data missing |
| fig2-60-fold-co-counter | Blocked: data missing |
| fig3-1hz-co-cross-same | Blocked: data missing |
| fig3-770khz-co-enhanced | Blocked: data missing |
| fig3-770khz-cross-same-as-1hz | Blocked: data missing |
| fig4a-single-pulse-follows-field | Reproduced (assumed: which base electrode was pulsed) |
| fig4bc-paired-pulse-focal-apex | Reproduced |
| fig4-comet-shape | Reproduced (assumed: a maximum robust to isolated bright objects) |
| fig5-base-sawtooth | Blocked: data missing |
| fig5-apex-larger-polarization | Blocked: data missing |

**How the analysis differs from the paper's.** The field map is computed here by a
two-dimensional finite-difference solve, in place of the paper's Sim4Life model; the electrode
positions are measured in the images and the scale is set from the 1.5 mm needles. The measures
for the image-only findings of Figure 4 are the claims file's readings, as the paper gives none.
Which base electrode was pulsed in Figure 4A is assumed (the right-hand one in the file's image).

**What was found about the dataset.** It holds only the two images of Figure 4; the data behind
Figures 2, 3 and 5, on which the paper's quantitative findings rest, are not deposited. The image
is stored as an `ImageSeries` whose first axis NWB reads as time, with no pixel size, channel
names or intensity unit. The session description of the single-pulse file gives the settings of a
Figure 3 experiment (co-directional pairs, 770 kHz, 10 kV/cm), and that of the paired-pulse file
an amplitude of 2 kV where the paper gives 4 kV. The file does not say which base electrode is E1.